# Inoculation Probing: MVP Experiment

**Can inoculation prompts disentangle spurious correlations in frozen LLMs?**

This notebook runs the minimal viable experiment to test whether prompts can improve probe generalization.

## 🔧 Setup

In [ ]:
# Check if running in Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
    
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Clone or navigate to repository
    import os
    repo_path = '/content/drive/MyDrive/inoculation-probing'
    
    if not os.path.exists(repo_path):
        print("Cloning repository to Google Drive...")
        !git clone https://github.com/mahadikprasad15/inoculation-probing.git $repo_path
    
    os.chdir(repo_path)
    print(f"Working directory: {os.getcwd()}")
else:
    print("Running locally")

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# HuggingFace authentication (required for Llama 3.2-1B)
if IN_COLAB:
    from huggingface_hub import notebook_login
    notebook_login()
else:
    print("Please run: huggingface-cli login")

## 📦 Imports

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

    import os
    if IN_COLAB:
        sys.path.insert(0, str(Path.cwd()))
    else:
        # Robustly find the correct src directory (inoculation-probing/src)
        current_path = Path.cwd()
        root_path = None

        # Check if we are in inoculation-probing root or notebooks
        if (current_path / 'src' / 'extraction').exists():
            root_path = current_path
        elif (current_path.parent / 'src' / 'extraction').exists():
            root_path = current_path.parent
        # Check if we are in parent directory (Probing-experiments)
        elif (current_path / 'inoculation-probing' / 'src' / 'extraction').exists():
            root_path = current_path / 'inoculation-probing'

        if root_path:
            if str(root_path) not in sys.path:
                sys.path.insert(0, str(root_path))
            print(f"Added {root_path} to sys.path")
        else:
            print("Warning: Could not find inoculation-probing/src directory")

print("✓ All imports successful")
from src.data import SpuriousDatasetBuilder
from src.extraction import ActivationExtractor
from src.probing import ProbeExperiment
from src.prompts import format_prompt, get_mvp_prompts, PROMPT_TEMPLATES
from src.utils import ExperimentLogger

print("✓ All imports successful")

## ⚙️ Configuration

In [ ]:
# Experiment configuration
CONFIG = {
    # Dataset
    'n_train_per_group': 100,  # 100 short+pos, 100 long+neg
    'n_test_per_group': 50,    # 50 short+neg, 50 long+pos
    'short_range': (10, 30),   # Token count for "short"
    'long_range': (100, 200),  # Token count for "long"
    
    # Model
    'model_name': 'meta-llama/Llama-3.2-1B',
    'layers': [12, 20],  # Middle and late layer
    
    # Experiment
    'seed': 42,
    'use_cache': True,
    
    # Paths
    'base_dir': Path('./results') if not IN_COLAB else Path('/content/drive/MyDrive/inoculation-probing/results')
}

# Get prompts to test
PROMPTS = get_mvp_prompts()

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")
print(f"\nPrompts to test: {PROMPTS}")

## 📊 Step 1: Create Spuriously Correlated Dataset

In [ ]:
dataset_dir = CONFIG['base_dir'] / 'datasets' / 'mvp_dataset'

if (dataset_dir / 'metadata.json').exists():
    print(f"Loading existing dataset from {dataset_dir}")
    dataset = SpuriousDatasetBuilder.load(dataset_dir)
else:
    print("Creating new spuriously correlated dataset...")
    dataset = SpuriousDatasetBuilder(
        source_dataset='imdb',
        train_correlation=1.0,  # Perfect spurious correlation
        n_train_per_group=CONFIG['n_train_per_group'],
        n_test_per_group=CONFIG['n_test_per_group'],
        short_range=CONFIG['short_range'],
        long_range=CONFIG['long_range'],
        seed=CONFIG['seed']
    )
    
    splits = dataset.create_splits()
    dataset.save(dataset_dir)

train_data = dataset.train_data
test_data = dataset.test_data

print(f"\n✓ Dataset ready")
print(f"  Train: {len(train_data['texts'])} samples")
print(f"  Test: {len(test_data['texts'])} samples")

### Visualize Dataset Distribution

In [ ]:
# Plot distribution
from collections import Counter

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Train distribution
train_counts = Counter(train_data['groups'])
axes[0].bar(train_counts.keys(), train_counts.values(), color='steelblue', alpha=0.7)
axes[0].set_title('Train Set (Spurious Correlation)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Test distribution
test_counts = Counter(test_data['groups'])
axes[1].bar(test_counts.keys(), test_counts.values(), color='coral', alpha=0.7)
axes[1].set_title('Test Set (Anti-correlated)')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\nTrain set has PERFECT spurious correlation (short→positive, long→negative)")
print("Test set is ANTI-CORRELATED (short→negative, long→positive)")
print("\n→ A probe that learns 'length' will fail on the test set!")

### Inspect Sample Texts

In [ ]:
print("Sample short positive review (train):")
short_pos_idx = train_data['groups'].index('short_positive')
print(f"  Length: {len(train_data['texts'][short_pos_idx].split())} tokens")
print(f"  Text: {train_data['texts'][short_pos_idx][:200]}...")

print("\nSample long negative review (train):")
long_neg_idx = train_data['groups'].index('long_negative')
print(f"  Length: {len(train_data['texts'][long_neg_idx].split())} tokens")
print(f"  Text: {train_data['texts'][long_neg_idx][:200]}...")

print("\nSample short negative review (test):")
short_neg_idx = test_data['groups'].index('short_negative')
print(f"  Length: {len(test_data['texts'][short_neg_idx].split())} tokens")
print(f"  Text: {test_data['texts'][short_neg_idx][:200]}...")

## 🔬 Step 2: Run Experiments

In [ ]:
# Initialize paths
activations_dir = CONFIG['base_dir'] / 'activations'
results_dir = CONFIG['base_dir'] / 'experiments' / 'mvp_experiment'

# Initialize logger
logger = ExperimentLogger(results_dir)

# Store all results
all_results = []

print("Starting experiments...\n")
print("="*80)

### Run Experiments for Each Layer and Prompt

In [ ]:
for layer in CONFIG['layers']:
    print(f"\n{'='*80}")
    print(f"LAYER {layer}")
    print(f"{'='*80}")
    
    # Initialize extractor for this layer
    extractor = ActivationExtractor(
        model_name=CONFIG['model_name'],
        layer=layer,
        cache_dir=activations_dir if CONFIG['use_cache'] else None
    )
    
    for prompt_name in PROMPTS:
        print(f"\n{'-'*80}")
        print(f"Prompt: {prompt_name}")
        print(f"{'-'*80}")
        
        # Show prompt template
        print(f"Template: {PROMPT_TEMPLATES[prompt_name][:100]}...")
        
        # Start logging run
        run_name = f"layer{layer}_{prompt_name}"
        config = {
            'layer': layer,
            'prompt_name': prompt_name,
            'model_name': CONFIG['model_name'],
            'n_train': len(train_data['texts']),
            'n_test': len(test_data['texts']),
            'seed': CONFIG['seed']
        }
        logger.start_run(run_name, config)
        
        # Format texts with prompt
        train_texts_prompted = [
            format_prompt(prompt_name, text)
            for text in train_data['texts']
        ]
        test_texts_prompted = [
            format_prompt(prompt_name, text)
            for text in test_data['texts']
        ]
        
        # Extract activations
        print("Extracting activations...")
        activations_train = extractor.extract(
            train_texts_prompted,
            prompt_name=f"{prompt_name}_train",
            use_cache=CONFIG['use_cache']
        )
        activations_test = extractor.extract(
            test_texts_prompted,
            prompt_name=f"{prompt_name}_test",
            use_cache=CONFIG['use_cache']
        )
        
        # Create probe experiment
        metadata_test = {
            'length_labels': test_data['length_labels'],
            'sentiment_labels': test_data['sentiment_labels']
        }
        
        probe_exp = ProbeExperiment(
            activations_train=activations_train,
            labels_train=np.array(train_data['sentiment_labels']),
            activations_test=activations_test,
            labels_test=np.array(test_data['sentiment_labels']),
            metadata_test=metadata_test
        )
        
        # Train probe
        probe_exp.train_probe(regularization='cv')
        
        # Evaluate
        metrics = probe_exp.evaluate(n_bootstrap=100)
        
        # Visualize
        viz_path = results_dir / f"viz_{run_name}.png"
        probe_exp.visualize_results(save_path=viz_path)
        
        # Display visualization
        from IPython.display import Image, display
        display(Image(filename=str(viz_path)))
        
        # Save probe
        probe_dir = results_dir / 'probes' / run_name
        probe_exp.save(probe_dir)
        
        # Log results
        logger.log_metrics(metrics)
        logger.log_artifact('visualization', viz_path)
        logger.log_artifact('probe', probe_dir / 'probe_model.pkl')
        logger.end_run()
        
        # Store for summary
        all_results.append({
            'layer': layer,
            'prompt': prompt_name,
            **metrics
        })

print("\n" + "="*80)
print("ALL EXPERIMENTS COMPLETE")
print("="*80)

## 📊 Step 3: Analyze Results

In [ ]:
import pandas as pd

# Create results dataframe
df_results = pd.DataFrame(all_results)

# Display results table
display_cols = ['layer', 'prompt', 'accuracy_overall', 'accuracy_worst_group']
print("\nResults Summary:")
print(df_results[display_cols].to_string(index=False))

# Save to CSV
csv_path = results_dir / 'results_summary.csv'
df_results.to_csv(csv_path, index=False)
print(f"\n✓ Saved results to {csv_path}")

### Visualize Comparison Across Prompts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, layer in enumerate(CONFIG['layers']):
    layer_results = df_results[df_results['layer'] == layer]
    
    ax = axes[i]
    
    # Plot overall and worst-group accuracy
    x = np.arange(len(layer_results))
    width = 0.35
    
    ax.bar(x - width/2, layer_results['accuracy_overall'], width, 
           label='Overall', alpha=0.7, color='steelblue')
    ax.bar(x + width/2, layer_results['accuracy_worst_group'], width,
           label='Worst-Group', alpha=0.7, color='coral')
    
    ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random')
    
    ax.set_xlabel('Prompt')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'Layer {layer}')
    ax.set_xticks(x)
    ax.set_xticklabels(layer_results['prompt'], rotation=45, ha='right')
    ax.legend()
    ax.set_ylim([0, 1.0])
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(results_dir / 'comparison_plot.png', dpi=150, bbox_inches='tight')
plt.show()

### Inoculation Effect Analysis

In [ ]:
print("\n" + "="*80)
print("INOCULATION EFFECT ANALYSIS")
print("="*80)

for layer in CONFIG['layers']:
    layer_results = df_results[df_results['layer'] == layer]
    
    baseline = layer_results[layer_results['prompt'] == 'baseline'].iloc[0]
    baseline_acc = baseline['accuracy_worst_group']
    
    print(f"\nLayer {layer}:")
    print(f"  Baseline (worst-group): {baseline_acc:.4f}")
    print(f"\n  Inoculation Effects:")
    
    for _, row in layer_results.iterrows():
        if row['prompt'] == 'baseline':
            continue
        
        prompt_acc = row['accuracy_worst_group']
        improvement = prompt_acc - baseline_acc
        
        symbol = "✓" if improvement > 0.05 else "→" if improvement > -0.05 else "✗"
        print(f"    {symbol} {row['prompt']:<25}: {prompt_acc:.4f} ({improvement:+.4f})")

print("\n" + "="*80)
print("\nInterpretation:")
print("  ✓ = Improvement > 5% (inoculation working!)")
print("  → = Within ±5% (no clear effect)")
print("  ✗ = Degradation > 5% (inoculation harmful)")

### Best Prompts

In [ ]:
print("\n" + "="*80)
print("BEST PROMPTS BY LAYER")
print("="*80)

for layer in CONFIG['layers']:
    layer_results = df_results[df_results['layer'] == layer]
    best = layer_results.loc[layer_results['accuracy_worst_group'].idxmax()]
    
    print(f"\nLayer {layer}:")
    print(f"  Best prompt: {best['prompt']}")
    print(f"  Overall accuracy: {best['accuracy_overall']:.4f}")
    print(f"  Worst-group accuracy: {best['accuracy_worst_group']:.4f}")

## 🎯 Step 4: Go/No-Go Decision

In [ ]:
print("\n" + "="*80)
print("GO/NO-GO DECISION")
print("="*80)

# Calculate maximum improvement
max_improvements = []

for layer in CONFIG['layers']:
    layer_results = df_results[df_results['layer'] == layer]
    baseline_acc = layer_results[layer_results['prompt'] == 'baseline']['accuracy_worst_group'].values[0]
    max_acc = layer_results['accuracy_worst_group'].max()
    improvement = max_acc - baseline_acc
    max_improvements.append(improvement)

best_improvement = max(max_improvements)

print(f"\nMaximum improvement across all conditions: {best_improvement:.4f} ({best_improvement*100:.1f}%)")
print("\nDecision criteria:")
print("  GO (>10% improvement): Proceed to mechanistic analysis")
print("  MAYBE (5-10% improvement): Collect more data or try more prompts")
print("  NO-GO (<5% improvement): Inoculation prompts don't help frozen models")

if best_improvement > 0.10:
    decision = "✓ GO"
    recommendation = "Strong signal! Proceed with mechanistic interpretability analysis."
elif best_improvement > 0.05:
    decision = "? MAYBE"
    recommendation = "Weak signal. Consider collecting more data or testing additional prompts."
else:
    decision = "✗ NO-GO"
    recommendation = "No clear signal. Inoculation prompts alone may not be sufficient for frozen models."

print(f"\n{decision}: {recommendation}")
print("\n" + "="*80)

## 📝 Conclusion

Review the results above to determine:

1. **Did inoculation prompts improve worst-group accuracy?**
2. **Which prompts worked best?**
3. **Did effects differ across layers?**
4. **Should we proceed to mechanistic analysis?**

---

**Next Steps:**

- **If GO**: Run mechanistic analysis (attention patterns, layer-wise probing, activation patching)
- **If MAYBE**: Increase dataset size, test more prompts, try different models
- **If NO-GO**: Document negative results, explore why prompts don't help frozen models

All results are saved to: `{results_dir}`